## SQL ⇄ pandas quick reference

| SQL | pandas |
|---|---|
| `SELECT a, b` | `df[['a','b']]` |
| `WHERE x > 100` | `df[df['x'] > 100]` |
| `WHERE a = 1 AND b = 2` | `df[(df['a']==1) & (df['b']==2)]` |
| `GROUP BY a` | `df.groupby('a', as_index=False)` |
| `GROUP BY a, b` | `df.groupby(['a','b'], as_index=False)` |
| `SUM(x) AS total` | `.agg(total=('x','sum'))` |
| `COUNT(*)` | `.size()` |
| `COUNT(DISTINCT x)` | `.agg(n=('x','nunique'))` |
| `HAVING SUM(x) > 100` | aggregate first, then mask the result |
| `ORDER BY x DESC` | `.sort_values('x', ascending=False)` |
| `LIMIT 10` | `.head(10)` |
| `CASE WHEN c THEN a ELSE b END` | `np.where(c, a, b)` |
| `LEFT JOIN` | `.merge(r, on='k', how='left')` |
| `INNER JOIN` | `.merge(r, on='k', how='inner')` |
| `FULL OUTER JOIN` | `.merge(r, on='k', how='outer')` |
| join on differently-named keys | `left_on='a', right_on='b'` |
| `COALESCE(x, 0)` | `.fillna(0)` |
| `PIVOT` | `pd.pivot_table(df, index=, columns=, values=, aggfunc=)` |
| UNPIVOT | `.melt(id_vars=, var_name=, value_name=)` |
| `UNION ALL` | `pd.concat([a, b])` |

**Gotchas**
- `merge` defaults to `how='inner'`. Always type `how=` explicitly.
- `groupby` puts keys in the index unless you pass `as_index=False`.
- Always check row count after a join — silent row multiplication means your right table wasn't unique on the key.
- Compute ratios *after* aggregating, never by averaging row-level ratios.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

df.shape

(9994, 21)

## Q1 — Aggregate by one dimension
```sql
SELECT Category, SUM(Sales) AS total_sales, SUM(Profit) AS total_profit
FROM orders
GROUP BY Category
ORDER BY total_sales DESC;
```

In [2]:
q1 = (df.groupby('Category', as_index=False)
        .agg(total_sales=('Sales','sum'),
             total_profit=('Profit','sum'))
        .sort_values('total_sales', ascending=False))
q1

,Category,total_sales,total_profit
2,Technology,836154.0330,145454.9481
0,Furniture,741999.7953,18451.2728
1,Office Supplies,719047.0320,122490.8008


## Q2 — Filter, then aggregate
```sql
SELECT Region, SUM(Sales) AS total_sales
FROM orders
WHERE Order_Date >= '2017-01-01'
GROUP BY Region;
```

In [3]:
q2 = (df[df['Order Date'] >= '2017-01-01']
        .groupby('Region', as_index=False)
        .agg(total_sales=('Sales','sum')))
q2

,Region,total_sales
0,Central,147098.1282
1,East,213082.9040
2,South,122905.8575
3,West,250128.3655


## Q3 — HAVING
```sql
SELECT Sub_Category, SUM(Sales) AS total_sales
FROM orders
GROUP BY Sub_Category
HAVING SUM(Sales) > 100000;
```

In [4]:
g = (df.groupby('Sub-Category', as_index=False)
       .agg(total_sales=('Sales','sum')))

q3 = g[g['total_sales'] > 100000].sort_values('total_sales', ascending=False)
q3

,Sub-Category,total_sales
13,Phones,330007.0540
5,Chairs,328449.1030
14,Storage,223843.6080
16,Tables,206965.5320
3,Binders,203412.7330
11,Machines,189238.6310
0,Accessories,167380.3180
6,Copiers,149528.0300
4,Bookcases,114879.9963
1,Appliances,107532.1610


## Q4 — Group by two dimensions
```sql
SELECT Region, Segment,
       SUM(Sales) AS total_sales,
       SUM(Profit) / SUM(Sales) AS margin
FROM orders
GROUP BY Region, Segment;
```

In [5]:
q4 = (df.groupby(['Region','Segment'], as_index=False)
        .agg(total_sales=('Sales','sum'),
             total_profit=('Profit','sum')))
q4['margin'] = q4['total_profit'] / q4['total_sales']
q4

,Region,Segment,total_sales,total_profit,margin
0,Central,Consumer,252031.4340,8564.0481,0.033980
1,Central,Corporate,157995.8128,18703.9020,0.118382
2,Central,Home Office,91212.6440,12438.4124,0.136367
3,East,Consumer,350908.1670,41190.9843,0.117384
4,East,Corporate,200409.3470,23622.5789,0.117872
5,East,Home Office,127463.7260,26709.2168,0.209544
6,South,Consumer,195580.9710,26913.5728,0.137608
7,South,Corporate,121885.9325,15215.2232,0.124832
8,South,Home Office,74255.0015,4620.6343,0.062227
9,West,Consumer,362880.7730,57450.6040,0.158318


## Q5 — Distinct counts and conditional logic
```sql
SELECT State,
       COUNT(DISTINCT Order_ID) AS order_count,
       SUM(Profit) AS total_profit,
       CASE WHEN SUM(Profit) < 0 THEN 'Loss' ELSE 'Profit' END AS flag
FROM orders
GROUP BY State;
```

In [6]:
q5 = (df.groupby('State', as_index=False)
        .agg(order_count=('Order ID','nunique'),
             total_profit=('Profit','sum')))
q5['flag'] = np.where(q5['total_profit'] < 0, 'Loss', 'Profit')
q5.sort_values('total_profit')

,State,order_count,total_profit,flag
41,Texas,487,-25729.3563,Loss
33,Ohio,236,-16971.3766,Loss
36,Pennsylvania,288,-15559.9603,Loss
11,Illinois,276,-12607.8870,Loss
31,North Carolina,136,-7490.9122,Loss
4,Colorado,79,-6527.8579,Loss
40,Tennessee,91,-5341.6936,Loss
1,Arizona,108,-3427.9246,Loss
8,Florida,200,-3399.3017,Loss
35,Oregon,56,-1190.4705,Loss


## Step 3 — Building a second table to join

The Superstore CSV has no Returns table (that lives in the Tableau .xls version),
so I'm fabricating one: a 5% random sample of Order IDs flagged as returned.
Also building a small Region → Regional Manager lookup as a dimension table.

In [7]:
returns = (df[['Order ID']]
             .drop_duplicates()
             .sample(frac=0.05, random_state=42)
             .assign(Returned='Yes'))

print(returns.shape)
returns.head()

(250, 2)


,Order ID,Returned
5058,CA-2016-120796,Yes
7069,CA-2014-141901,Yes
8575,CA-2017-101273,Yes
3242,CA-2017-114524,Yes
6761,CA-2016-162943,Yes


In [8]:
managers = pd.DataFrame({
    'Region': ['West', 'East', 'Central', 'South'],
    'Manager': ['Sadie Pawthorne', 'Chuck Magee', 'Roxanne Rodriguez', 'Fred Suzuki']
})
managers

,Region,Manager
0,West,Sadie Pawthorne
1,East,Chuck Magee
2,Central,Roxanne Rodriguez
3,South,Fred Suzuki


In [9]:
orders_flagged = df.merge(returns, on='Order ID', how='left')

print('before:', df.shape)
print('after: ', orders_flagged.shape)
orders_flagged['Returned'].value_counts(dropna=False)

before: (9994, 21)
after:  (9994, 22)


Returned
NaN    9489
Yes     505
Name: count, dtype: int64

In [10]:
orders_flagged['Returned'] = orders_flagged['Returned'].fillna('No')
orders_flagged['Returned'].value_counts()

Returned
No     9489
Yes     505
Name: count, dtype: int64

In [11]:
for how in ['left', 'inner', 'outer', 'right']:
    result = df.merge(returns, on='Order ID', how=how)
    print(f"{how:>6}: {result.shape[0]:>6,} rows")

  left:  9,994 rows
 inner:    505 rows
 outer:  9,994 rows
 right:    505 rows


In [12]:
check = df.merge(returns, on='Order ID', how='outer', indicator=True)
check['_merge'].value_counts()

_merge
left_only     9489
both           505
right_only       0
Name: count, dtype: int64

In [13]:
orders_flagged = orders_flagged.merge(managers, on='Region', how='left')
orders_flagged[['Order ID','Region','Manager','Returned']].head()

,Order ID,Region,Manager,Returned
0,CA-2016-152156,South,Fred Suzuki,No
1,CA-2016-152156,South,Fred Suzuki,No
2,CA-2016-138688,West,Sadie Pawthorne,No
3,US-2015-108966,South,Fred Suzuki,No
4,US-2015-108966,South,Fred Suzuki,No


In [14]:
pivot = pd.pivot_table(orders_flagged,
                       index='Sub-Category',
                       columns='Region',
                       values='Sales',
                       aggfunc='sum',
                       margins=True,
                       margins_name='Total')
pivot.round(0)

Region,Central,East,South,West,Total
Sub-Category,,,,,
Accessories,33956.0,45033.0,27277.0,61114.0,167380.0
Appliances,23582.0,34188.0,19525.0,30236.0,107532.0
Art,5765.0,7486.0,4656.0,9212.0,27119.0
Binders,56923.0,53498.0,37030.0,55961.0,203413.0
Bookcases,24157.0,43819.0,10899.0,36004.0,114880.0
Chairs,85231.0,96261.0,45176.0,101781.0,328449.0
Copiers,37260.0,53219.0,9300.0,49749.0,149528.0
Envelopes,4637.0,4376.0,3346.0,4118.0,16476.0
Fasteners,778.0,820.0,503.0,923.0,3024.0


In [15]:
orders_flagged['Is Returned'] = (orders_flagged['Returned'] == 'Yes').astype(int)

return_rate = pd.pivot_table(orders_flagged,
                             index='Category',
                             columns='Region',
                             values='Is Returned',
                             aggfunc='mean',
                             margins=True,
                             margins_name='Total')
(return_rate * 100).round(1)

Region,Central,East,South,West,Total
Category,,,,,
Furniture,4.8,5.5,6.3,5.4,5.4
Office Supplies,4.4,4.3,5.2,4.3,4.5
Technology,5.5,5.8,5.5,8.2,6.4
Total,4.7,4.8,5.5,5.3,5.1


In [16]:
lost = pd.pivot_table(orders_flagged[orders_flagged['Is Returned'] == 1],
                      index='Category',
                      columns='Region',
                      values='Profit',
                      aggfunc='sum',
                      margins=True,
                      margins_name='Total')
lost.round(0)

Region,Central,East,South,West,Total
Category,,,,,
Furniture,-467.0,1302.0,1469.0,1745.0,4050.0
Office Supplies,-4840.0,3976.0,1089.0,1484.0,1709.0
Technology,705.0,3155.0,2350.0,2482.0,8691.0
Total,-4603.0,8433.0,4908.0,5711.0,14450.0


In [17]:
long = (return_rate
          .drop(index='Total', columns='Total')
          .reset_index()
          .melt(id_vars='Category',
                var_name='Region',
                value_name='return_rate'))
long

,Category,Region,return_rate
0,Furniture,Central,0.047817
1,Office Supplies,Central,0.044304
2,Technology,Central,0.054762
3,Furniture,East,0.054908
4,Office Supplies,East,0.043224
5,Technology,East,0.057944
6,Furniture,South,0.063253
7,Office Supplies,South,0.052261
8,Technology,South,0.054608
9,Furniture,West,0.053748
